# ML Assignment 4 – Regression Evaluation Metrics

**Dataset:** California Housing dataset  
**Objective:** Implement regression models, evaluate them using MSE, MAE and R², perform 5-fold cross-validation, tune hyperparameters, and select the best model.

This notebook is prepared as a runnable submission with answers and explanations.

## 1. Data Loading and Preprocessing

The California Housing dataset is loaded using `fetch_california_housing` from scikit-learn and converted into a pandas DataFrame.

For preprocessing:
- Missing values are checked.
- Standardization is used because feature magnitudes differ considerably and it is especially useful for SVR and linear models.
- Tree-based models do not require scaling, but the standardized data is also used consistently for the requested model comparison.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Basic dataset information
print("Missing values:")
display(df.isnull().sum())

print("\nDataset information:")
df.info()


In [ ]:
# Descriptive statistics
display(df.describe())


### Exploratory Data Analysis (EDA)

The following plots help understand feature distributions and the relationship among variables.

In [ ]:
# Histograms
df.hist(figsize=(14, 10), bins=30)
plt.suptitle("Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
plt.imshow(df.corr(), cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(df.columns)), df.columns, rotation=90)
plt.yticks(range(len(df.columns)), df.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


### Preprocessing Justification

The dataset is checked for missing values before modeling. An imputer is included in the modeling pipeline so that the workflow remains robust if missing values are present. StandardScaler transforms features to approximately zero mean and unit variance. This is important for SVR because distance-based calculations are sensitive to feature scale, and it also helps linear models train on features with different units.

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 2. Regression Algorithm Implementation

The required algorithms are:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Gradient Boosting Regressor
5. Support Vector Regressor (SVR)

Each model is placed in a pipeline with missing-value handling and standardization.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Decision Tree Regressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "Random Forest Regressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ]),

    "Gradient Boosting Regressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),

    "SVR": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR())
    ])
}

for name in models:
    print(name)


### Brief Explanation of the Algorithms

**Linear Regression:** Fits a linear relationship between the input features and the target. It is simple, interpretable, and provides a useful baseline.

**Decision Tree Regressor:** Splits the feature space into regions using decision rules. It can model nonlinear relationships without requiring a linear assumption.

**Random Forest Regressor:** Combines many decision trees and averages their predictions. This generally reduces overfitting compared with a single decision tree.

**Gradient Boosting Regressor:** Builds trees sequentially, with each new tree attempting to improve the errors of the previous ensemble. It can capture complex nonlinear patterns.

**SVR:** Finds a regression function using support vectors and an epsilon-insensitive loss. With an RBF kernel, it can model nonlinear relationships and benefits from feature scaling.

In [ ]:
# Train all models and evaluate on the test set
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MSE": mse,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
display(results_df)


### Model Comparison

For MSE and MAE, lower values indicate lower prediction error. For R², a higher value indicates that the model explains more of the variation in the target.

The table above provides the actual test-set comparison produced when the notebook is run.

In [ ]:
# Identify the best and worst models based on R²
best_model_name = results_df.loc[results_df["R2"].idxmax(), "Model"]
worst_model_name = results_df.loc[results_df["R2"].idxmin(), "Model"]

print("Best-performing model based on test R²:", best_model_name)
print("Worst-performing model based on test R²:", worst_model_name)

print("\nJustification:")
print("The best model has the highest R² while also being considered alongside its MSE and MAE.")
print("The worst model has the lowest R² among the evaluated models.")


## 4. 5-Fold Cross-Validation

Five-fold cross-validation is used to evaluate model performance across multiple training/validation splits. This provides a more robust estimate than relying on a single train-test split.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=kf,
        scoring={
            "MSE": "neg_mean_squared_error",
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV MSE Mean": -scores["test_MSE"].mean(),
        "CV MAE Mean": -scores["test_MAE"].mean(),
        "CV R2 Mean": scores["test_R2"].mean(),
        "CV R2 Std": scores["test_R2"].std()
    })

cv_df = pd.DataFrame(cv_results).sort_values("CV R2 Mean", ascending=False).reset_index(drop=True)
display(cv_df)


## 5. Hyperparameter Tuning

GridSearchCV is used to tune important hyperparameters. The selected parameters are deliberately kept manageable so that the notebook remains practical to run.

The tuned parameters are:
- Decision Tree: `max_depth`, `min_samples_split`
- Random Forest: `n_estimators`, `max_depth`, `min_samples_split`
- Gradient Boosting: `n_estimators`, `learning_rate`, `max_depth`
- SVR: `C`, `gamma`

Linear Regression does not have a comparable set of major tree/kernel hyperparameters, so it is retained as the baseline model.

In [ ]:
param_grids = {
    "Decision Tree Regressor": {
        "model__max_depth": [10, 20, None],
        "model__min_samples_split": [2, 5]
    },

    "Random Forest Regressor": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 20],
        "model__min_samples_split": [2, 5]
    },

    "Gradient Boosting Regressor": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3]
    },

    "SVR": {
        "model__C": [10, 100],
        "model__gamma": ["scale", 0.01]
    }
}

tuned_models = {}
tuning_results = []

for name, grid in param_grids.items():
    print(f"Tuning {name}...")

    grid_search = GridSearchCV(
        models[name],
        grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    tuned_models[name] = grid_search.best_estimator_

    tuning_results.append({
        "Model": name,
        "Best CV R2": grid_search.best_score_,
        "Best Parameters": grid_search.best_params_
    })

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df)


In [ ]:
# Evaluate tuned models on the held-out test set
tuned_test_results = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)

    tuned_test_results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

tuned_test_df = pd.DataFrame(tuned_test_results).sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(tuned_test_df)


### Impact of Hyperparameter Tuning

Hyperparameters control model complexity and learning behavior. For example:
- Increasing tree depth can allow a model to learn more complex patterns but can also increase overfitting.
- `n_estimators` controls the number of trees in ensemble models.
- `learning_rate` controls how strongly each boosting stage contributes.
- `C` and `gamma` control the flexibility of SVR.

The actual impact should be determined from the cross-validation and test results generated above rather than assumed in advance.

## 6. Selecting the Best Regression Model

The final model is selected using the evaluation metrics, cross-validation performance, and tuned test performance.

A model with a high R² together with low MSE and MAE provides strong regression performance. Cross-validation is also considered to check whether the performance is consistent across folds.

In [ ]:
# Combine tuned models with the baseline Linear Regression for final comparison
final_candidates = {
    "Linear Regression": models["Linear Regression"],
    **tuned_models
}

final_results = []

for name, model in final_candidates.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    final_results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred)
    })

final_df = pd.DataFrame(final_results).sort_values(
    ["R2", "MSE", "MAE"],
    ascending=[False, True, True]
).reset_index(drop=True)

display(final_df)

selected_model = final_df.iloc[0]["Model"]

print("Selected best model based on the final evaluation:", selected_model)
print("\nReason:")
print("It has the strongest overall combination of R², MSE and MAE in the final comparison.")
print("Its cross-validation performance should also be checked above to confirm robustness.")


## Conclusion

The California Housing dataset was loaded and preprocessed, followed by implementation of Linear Regression, Decision Tree, Random Forest, Gradient Boosting, and SVR models.

The models were evaluated using:
- Mean Squared Error (MSE)
- Mean Absolute Error (MAE)
- R-squared (R²)

Five-fold cross-validation was performed, and GridSearchCV was used for hyperparameter tuning of the nonlinear models. The final selection is based on the measured results generated by the notebook, considering both prediction errors and R².